# PrahaarShield – Text Engine v3 training (Colab, T4 GPU, ~1 hour)
Trains **M1 v3** (MuRIL multi-task message model) → **calibration** → evaluation → **M7** (conversation model) → evaluation.

**Drive layout** (`MyDrive/PrahaarShield/`):
```
prepared_v3/      train.jsonl val.jsonl test.jsonl            (from prepare_v3.py)
conversations/    conv_train.jsonl conv_val.jsonl conv_test.jsonl   (from build_conversations.py)
code/             engine/  common/   (from the app zip: same normalizer + placeholders as the app)
models_v3/        created automatically
```
Runtime → Change runtime type → **T4 GPU** → Run all. Copy the **SUMMARY** cell output to Claude.
Optional baseline: set `MULTITASK = False` and run again (skips M7) for the binary comparison.

In [ ]:
!nvidia-smi -L
!pip -q install transformers==4.44.2 scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/PrahaarShield'
import os, sys
sys.path.append(f'{ROOT}/code')
OUT = f'{ROOT}/models_v3'; os.makedirs(OUT, exist_ok=True)
print(os.listdir(f'{ROOT}/prepared_v3'), os.listdir(f'{ROOT}/conversations'))

In [ ]:
# ---------------- CONFIG ----------------
MULTITASK = True
CFG = dict(model='google/muril-base-cased', max_len=256, epochs=8, patience=2, lr_enc=3e-5, lr_head=1e-3, batch=16,
           warmup=0.1, seed=2026, placeholder_dropout=0.3, max_fpr=0.05, w_scam=1.0, w_type=1.0, w_pillar=1.0, w_lang=0.2)
TAG = 'muril_multitask' if MULTITASK else 'muril_binary'
import json, random, re, time, csv, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print(DEVICE, TAG)
SUMMARY = {}

In [ ]:
# ---------------- DATA ----------------
load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
train, val, test = [load(f'{ROOT}/prepared_v3/{s}.jsonl') for s in ('train', 'val', 'test')]
try:
    from engine.entities import prepare_for_model
    def to_model_text(t):
        n, e = prepare_for_model(t); return e.text_model
    print('engine found: normalizer + placeholders ON')
except Exception as ex:
    raise SystemExit('Upload code/engine and code/common to Drive first: ' + str(ex))
for rows in (train, val, test):
    for r in rows: r['text_model'] = to_model_text(r['text'])
PILLARS = ['P_IDENTITY','P_SITUATION','P_STAKES','P_CRED','P_RESOURCE','P_CONTROL']
TYPES = sorted({r['scam_type'] for r in train + val + test if r['is_scam'] == 1})
LANGS = sorted({r['language'] for r in train + val + test})
T2I = {t: i for i, t in enumerate(TYPES)}; L2I = {l: i for i, l in enumerate(LANGS)}
print(len(train), len(val), len(test), TYPES, LANGS)
g = np.mean([r['is_scam'] for r in train]); LW = {}
for l in LANGS:
    p = np.mean([r['is_scam'] for r in train if r['language'] == l] or [g]); p = min(max(p, .02), .98)
    LW[(l, 1)] = float(np.clip(g / p, .5, 3)); LW[(l, 0)] = float(np.clip((1 - g) / (1 - p), .5, 3))
CW = {1: len(train) / (2 * sum(r['is_scam'] for r in train)), 0: len(train) / (2 * sum(1 - r['is_scam'] for r in train))}
tc = np.array([sum(r['scam_type'] == t for r in train) for t in TYPES], dtype=float)
TYPE_W = torch.tensor(tc.sum() / (len(TYPES) * np.maximum(tc, 1)), dtype=torch.float)
known = [r for r in train if r.get('pillar_labels') is not None]
pc = np.array([sum(p in r['pillar_labels'] for r in known) for p in PILLARS], dtype=float)
PIL_POSW = torch.tensor(np.clip((len(known) - pc) / np.maximum(pc, 1), 1, 5), dtype=torch.float)
print('rows with unknown pillars (masked):', len(train) - len(known))

In [ ]:
# ---------------- DATASET ----------------
tok = AutoTokenizer.from_pretrained(CFG['model'])
PH = re.compile(r'<(URL|UPI_LINK|UPI|EMAIL|PHONE|PHONE_INTL|AMOUNT|ACCOUNT|AADHAAR|PAN|CODE)>')
class DS(Dataset):
    def __init__(self, rows, train=False): self.rows, self.train = rows, train
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]; t = r['text_model']
        if self.train and CFG['placeholder_dropout'] > 0:
            t = PH.sub(lambda m: '' if random.random() < CFG['placeholder_dropout'] else m.group(0), t)
        return t, r
def collate(batch):
    texts, rows = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=CFG['max_len'], padding=True, return_tensors='pt')
    y = torch.tensor([r['is_scam'] for r in rows])
    yt = torch.tensor([T2I[r['scam_type']] if r['is_scam'] == 1 and r['scam_type'] in T2I else -100 for r in rows])
    yp = torch.tensor([[1. if p in (r.get('pillar_labels') or []) else 0. for p in PILLARS] for r in rows])
    pm = torch.tensor([0. if r.get('pillar_labels') is None else 1. for r in rows])
    yl = torch.tensor([L2I[r['language']] for r in rows])
    w = torch.tensor([LW[(r['language'], r['is_scam'])] * CW[r['is_scam']] for r in rows], dtype=torch.float)
    return enc, y, yt, yp, pm, yl, w
dl = lambda rows, tr: DataLoader(DS(rows, tr), batch_size=CFG['batch'], shuffle=tr, collate_fn=collate)

In [ ]:
# ---------------- M1 MODEL (same structure as app/engine/text_engine.py) ----------------
class PrahaarModel(nn.Module):
    def __init__(self, name, n_types, n_langs):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name); h = self.enc.config.hidden_size
        self.drop = nn.Dropout(0.1)
        self.scam, self.stype = nn.Linear(h, 2), nn.Linear(h, n_types)
        self.pil, self.lang = nn.Linear(h, len(PILLARS)), nn.Linear(h, n_langs)
    def pooled(self, ids, mask):
        hs = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).float(); return (hs * m).sum(1) / m.sum(1).clamp(min=1)
    def forward(self, ids, mask, return_vec=False):
        v = self.pooled(ids, mask); x = self.drop(v)
        out = (self.scam(x), self.stype(x), self.pil(x), self.lang(x))
        return out + (v,) if return_vec else out
model = PrahaarModel(CFG['model'], len(TYPES), len(LANGS)).to(DEVICE)
ce = nn.CrossEntropyLoss(reduction='none'); ce_type = nn.CrossEntropyLoss(weight=TYPE_W.to(DEVICE), ignore_index=-100)
bce = nn.BCEWithLogitsLoss(pos_weight=PIL_POSW.to(DEVICE), reduction='none'); ce_lang = nn.CrossEntropyLoss()

@torch.no_grad()
def logits_of(rows):
    model.eval(); S, Ty, P, V = [], [], [], []
    for enc, *_ in dl(rows, False):
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            o_s, o_t, o_p, _, v = model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE), return_vec=True)
        S.append(o_s.float().cpu()); Ty.append(o_t.float().cpu()); P.append(o_p.float().cpu()); V.append(v.float().cpu())
    return torch.cat(S), torch.cat(Ty), torch.cat(P), torch.cat(V)

In [ ]:
# ---------------- TRAIN M1 ----------------
tr_dl = dl(train, True)
enc_p = list(model.enc.parameters()); head_p = [p for n, p in model.named_parameters() if not n.startswith('enc.')]
opt = torch.optim.AdamW([{'params': enc_p, 'lr': CFG['lr_enc']}, {'params': head_p, 'lr': CFG['lr_head']}], weight_decay=0.01)
steps = len(tr_dl) * CFG['epochs']; sch = get_linear_schedule_with_warmup(opt, int(steps * CFG['warmup']), steps)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == 'cuda'); best, bad = -1, 0
def type_f1(rows, t):
    idx = [k for k, r in enumerate(rows) if r['is_scam'] == 1 and r['scam_type'] in T2I]
    return f1_score([T2I[rows[k]['scam_type']] for k in idx], t[idx], average='macro') if idx else 0.0
for ep in range(CFG['epochs']):
    model.train(); t0 = time.time(); tot = 0
    for enc, y, yt, yp, pm, yl, w in tr_dl:
        ids, mask = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)
        y, yt, yp, pm, yl, w = y.to(DEVICE), yt.to(DEVICE), yp.to(DEVICE), pm.to(DEVICE), yl.to(DEVICE), w.to(DEVICE)
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            o_s, o_t, o_p, o_l = model(ids, mask)
        loss = CFG['w_scam'] * (ce(o_s.float(), y) * w).mean()
        if MULTITASK:
            if (yt != -100).any(): loss = loss + CFG['w_type'] * ce_type(o_t.float(), yt)
            if pm.sum() > 0: loss = loss + CFG['w_pillar'] * (bce(o_p.float(), yp).mean(1) * pm).sum() / pm.sum()
            loss = loss + CFG['w_lang'] * ce_lang(o_l.float(), yl)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
    S, Ty, _, _ = logits_of(val); p = torch.softmax(S, -1)[:, 1].numpy()
    f1 = f1_score([r['is_scam'] for r in val], (p >= .5).astype(int), average='macro')
    tf1 = type_f1(val, Ty.argmax(-1).numpy()) if MULTITASK else 0.0; score = f1 + (0.5 * tf1 if MULTITASK else 0)
    print(f'epoch {ep+1}: loss {tot/len(tr_dl):.4f} | val scam F1 {f1:.4f} | val type F1 {tf1:.4f} | {time.time()-t0:.0f}s')
    if score > best:
        best, bad = score, 0; torch.save(model.state_dict(), f'{OUT}/{TAG}.pt'); print('   saved best')
    else:
        bad += 1
        if bad >= CFG['patience']: print('early stop'); break
model.load_state_dict(torch.load(f'{OUT}/{TAG}.pt')); tok.save_pretrained(f'{OUT}/{TAG}_tokenizer')

In [ ]:
# ---------------- CALIBRATION (temperature scaling on VAL) + threshold ----------------
def fit_T(logits, target, kind):
    T = torch.ones(1, requires_grad=True); o = torch.optim.LBFGS([T], lr=0.05, max_iter=200)
    def step():
        o.zero_grad()
        if kind == 'bce': l = nn.functional.binary_cross_entropy_with_logits(logits / T.clamp(min=0.05), target)
        else: l = nn.functional.cross_entropy(logits / T.clamp(min=0.05), target)
        l.backward(); return l
    o.step(step); return float(T.clamp(min=0.05))
def ece(p, y, bins=10):
    p, y = np.asarray(p), np.asarray(y); e = 0
    for b in range(bins):
        m = (p >= b / bins) & (p < (b + 1) / bins) if b < bins - 1 else (p >= b / bins)
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)
Sv, Tyv, Pv, _ = logits_of(val); yv = torch.tensor([r['is_scam'] for r in val])
TEMPS = {'scam': fit_T(Sv, yv, 'ce')}
if MULTITASK:
    idx = [k for k, r in enumerate(val) if r['is_scam'] == 1 and r['scam_type'] in T2I]
    TEMPS['type'] = fit_T(Tyv[idx], torch.tensor([T2I[val[k]['scam_type']] for k in idx]), 'ce')
    km = [k for k, r in enumerate(val) if r.get('pillar_labels') is not None]
    TEMPS['pillar'] = fit_T(Pv[km], torch.tensor([[1. if p in val[k]['pillar_labels'] else 0. for p in PILLARS] for k in km]), 'bce')
pv_raw = torch.softmax(Sv, -1)[:, 1].numpy(); pv = torch.softmax(Sv / TEMPS['scam'], -1)[:, 1].numpy(); yvn = yv.numpy()
print('temperatures', TEMPS, '| ECE val before', round(ece(pv_raw, yvn), 4), 'after', round(ece(pv, yvn), 4))
best_th, best_f = 0.5, -1
for th in np.arange(0.05, 0.96, 0.01):
    pr = (pv >= th).astype(int); fpr = pr[yvn == 0].mean(); f = f1_score(yvn, pr, average='macro')
    if fpr <= CFG['max_fpr'] and f > best_f: best_th, best_f = float(th), f
print(f'threshold {best_th:.2f} (val macro-F1 {best_f:.3f})')
json.dump(dict(CFG=CFG, TYPES=TYPES, LANGS=LANGS, PILLARS=PILLARS, multitask=MULTITASK, threshold=best_th, temperatures=TEMPS, version='v3'),
          open(f'{OUT}/{TAG}_config.json', 'w'), indent=2)
SUMMARY['M1_temperatures'] = TEMPS; SUMMARY['M1_threshold'] = best_th

In [ ]:
# ---------------- EVALUATE M1 on TEST ----------------
S, Ty, P, _ = logits_of(test); y = np.array([r['is_scam'] for r in test])
p = torch.softmax(S / TEMPS['scam'], -1)[:, 1].numpy(); pred = (p >= best_th).astype(int); ben = y == 0
print(classification_report(y, pred, target_names=['benign', 'scam'], digits=3))
SUMMARY['M1_test'] = dict(macro_f1=round(f1_score(y, pred, average='macro'), 4), scam_f1=round(f1_score(y, pred), 4),
                          recall=round(float(pred[~ben].mean()), 4), false_alarm=round(float(pred[ben].mean()), 4), ece=round(ece(p, y), 4))
SUMMARY['M1_per_language'] = {l: round(f1_score(y[i], pred[i], average='macro', zero_division=0), 3)
                              for l in LANGS for i in [[k for k, r in enumerate(test) if r['language'] == l]] if len(i) >= 10}
if MULTITASK:
    t = Ty.argmax(-1).numpy(); SUMMARY['M1_type_macro_f1'] = round(type_f1(test, t), 4)
    km = [k for k, r in enumerate(test) if r.get('pillar_labels') is not None]
    yp = np.array([[1 if q in test[k]['pillar_labels'] else 0 for q in PILLARS] for k in km]); pp = (torch.sigmoid(P[km] / TEMPS['pillar']).numpy() >= .5).astype(int)
    SUMMARY['M1_pillar_f1'] = {q: round(f1_score(yp[:, j], pp[:, j], zero_division=0), 3) for j, q in enumerate(PILLARS)}
with open(f'{OUT}/{TAG}_test_errors.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['text', 'true', 'p_scam', 'scam_type', 'pred_type', 'language', 'category'])
    for k, r in enumerate(test):
        wrong = pred[k] != y[k] or (MULTITASK and r['is_scam'] == 1 and r['scam_type'] in T2I and TYPES[Ty[k].argmax()] != r['scam_type'])
        if wrong: w.writerow([r['text'], r['is_scam'], round(float(p[k]), 3), r['scam_type'], TYPES[int(Ty[k].argmax())], r['language'], r['category']])
print(json.dumps(SUMMARY, indent=1))

In [ ]:
# ---------------- M7: features for every conversation message (frozen M1) ----------------
if not MULTITASK: raise SystemExit('Binary run finished (M7 is trained only in the multi-task run).')
STAGES = ['none', 'approach', 'pressure', 'demand']
convs = {s: load(f'{ROOT}/conversations/conv_{s}.jsonl') for s in ('train', 'val', 'test')}
texts = sorted({m['text'] for s in convs for c in convs[s] for m in c['messages']})
rows = [{'text': t, 'text_model': to_model_text(t), 'is_scam': 0, 'scam_type': 'benign', 'language': LANGS[0], 'pillar_labels': []} for t in texts]
S, _, P, V = logits_of(rows)                        # RAW (uncalibrated) scores, exactly like the app's features()
FEAT = {t: torch.cat([V[i], torch.softmax(S[i], -1)[1:2], torch.sigmoid(P[i])]).numpy() for i, t in enumerate(texts)}
print('features for', len(FEAT), 'unique messages; dim', len(next(iter(FEAT.values()))))

In [ ]:
# ---------------- M7: conversation model (same class as app/engine/conversation.py) ----------------
FEAT_DIM, MAX_MSGS = 768 + 7, 8
def build_conv_model(n_types, d=256, layers=2, heads=4, max_msgs=MAX_MSGS):
    class ConvModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.inp = nn.Sequential(nn.LayerNorm(FEAT_DIM), nn.Linear(FEAT_DIM, d)); self.spk = nn.Embedding(2, d); self.pos = nn.Embedding(max_msgs, d)
            layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.1, batch_first=True); self.enc = nn.TransformerEncoder(layer, layers)
            self.scam, self.stage, self.stype = nn.Linear(d, 2), nn.Linear(d, len(STAGES)), nn.Linear(d, n_types)
        def forward(self, x, spk, pad):
            n = x.shape[1]; pos = torch.arange(n, device=x.device).unsqueeze(0).expand(x.shape[0], n)
            h = self.inp(x) + self.spk(spk) + self.pos(pos)
            causal = torch.triu(torch.ones(n, n, dtype=torch.bool, device=x.device), 1)
            h = self.enc(h, mask=causal, src_key_padding_mask=pad)
            return self.scam(h), self.stage(h), self.stype(h)
    return ConvModel()
def windows(chat):                                    # sliding windows of <= 8 messages, labels per position
    msgs, out = chat['messages'], []
    starts = range(0, max(1, len(msgs) - MAX_MSGS + 1))
    for s0 in starts:
        w = msgs[s0:s0 + MAX_MSGS]; so_far = any(m['is_scam_msg'] for m in msgs[:s0])
        X, spk, ys, yst, yty = [], [], [], [], []
        for m in w:
            so_far = so_far or bool(m['is_scam_msg'])
            X.append(FEAT[m['text']]); spk.append(0 if m['speaker'] == 'them' else 1); ys.append(int(so_far))
            yst.append(STAGES.index(m['stage'] or 'none') if m['speaker'] == 'them' else -100)
            yty.append(T2I[chat['scam_type']] if so_far and chat['scam_type'] in T2I else -100)
        out.append((np.stack(X), spk, ys, yst, yty))
    return out
def batchify(ws):
    n = max(len(w[1]) for w in ws); B = len(ws)
    X = torch.zeros(B, n, FEAT_DIM); spk = torch.zeros(B, n, dtype=torch.long); pad = torch.ones(B, n, dtype=torch.bool)
    ys, yst, yty = [torch.full((B, n), -100, dtype=torch.long) for _ in range(3)]
    for b, (x, s, a, st, ty) in enumerate(ws):
        k = len(s); X[b, :k] = torch.tensor(x); spk[b, :k] = torch.tensor(s); pad[b, :k] = False
        ys[b, :k] = torch.tensor(a); yst[b, :k] = torch.tensor(st); yty[b, :k] = torch.tensor(ty)
    return X, spk, pad, ys, yst, yty
W = {s: [w for c in convs[s] for w in windows(c)] for s in convs}
print({s: len(W[s]) for s in W})
conv = build_conv_model(len(TYPES)).to(DEVICE); copt = torch.optim.AdamW(conv.parameters(), lr=1e-3, weight_decay=0.01)
cel = nn.CrossEntropyLoss(ignore_index=-100); bestc, badc = 1e9, 0
for ep in range(30):
    conv.train(); random.shuffle(W['train']); tl = 0
    for i in range(0, len(W['train']), 32):
        X, spk, pad, ys, yst, yty = [t.to(DEVICE) for t in batchify(W['train'][i:i + 32])]
        a, b, c = conv(X, spk, pad)
        loss = cel(a.reshape(-1, 2), ys.reshape(-1)) + cel(b.reshape(-1, 4), yst.reshape(-1)) + 0.5 * cel(c.reshape(-1, len(TYPES)), yty.reshape(-1))
        copt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(conv.parameters(), 1.0); copt.step(); tl += loss.item()
    conv.eval(); vl = 0
    with torch.no_grad():
        for i in range(0, len(W['val']), 64):
            X, spk, pad, ys, yst, yty = [t.to(DEVICE) for t in batchify(W['val'][i:i + 64])]
            a, b, c = conv(X, spk, pad)
            vl += (cel(a.reshape(-1, 2), ys.reshape(-1)) + cel(b.reshape(-1, 4), yst.reshape(-1))).item()
    print(f'M7 epoch {ep+1}: train {tl:.2f} | val {vl:.3f}')
    if vl < bestc: bestc, badc = vl, 0; torch.save(conv.state_dict(), f'{OUT}/conv_model.pt')
    else:
        badc += 1
        if badc >= 4: print('early stop'); break
conv.load_state_dict(torch.load(f'{OUT}/conv_model.pt'))
json.dump(dict(TYPES=TYPES, STAGES=STAGES, max_msgs=MAX_MSGS, d=256, layers=2, heads=4, feat='M1 mean-pooled 768 + raw scam prob + 6 raw pillars'),
          open(f'{OUT}/conv_config.json', 'w'), indent=2)

In [ ]:
# ---------------- M7 evaluation: stream every TEST chat message by message ----------------
@torch.no_grad()
def stream(chat):
    msgs, out = chat['messages'], []
    for i, m in enumerate(msgs):
        if m['speaker'] != 'them': continue
        w = msgs[max(0, i + 1 - MAX_MSGS): i + 1]
        X = torch.tensor(np.stack([FEAT[x['text']] for x in w]))[None].to(DEVICE)
        spk = torch.tensor([[0 if x['speaker'] == 'them' else 1 for x in w]]).to(DEVICE)
        a, b, c = conv(X, spk, torch.zeros(1, len(w), dtype=torch.bool, device=DEVICE))
        out.append((i, float(torch.softmax(a[0, -1], -1)[1]), STAGES[int(b[0, -1].argmax())], m))
    return out
def conv_metrics(chats):
    r = dict(scam_chats=0, scam_detected=0, demand_chats=0, demand_caught=0, benign_chats=0, benign_alerted=0, slow=0, slow_early=0, stage_ok=0, stage_n=0)
    for ch in chats:
        s = stream(ch); is_scam = ch['kind'] in ('scam', 'slow_burn')
        alerted = [k for k, (i, p, st, m) in enumerate(s) if p >= 0.5]
        for i, p, st, m in s:
            r['stage_n'] += 1; r['stage_ok'] += int(st == (m['stage'] or 'none'))
        if is_scam:
            r['scam_chats'] += 1; r['scam_detected'] += int(bool(alerted))
            if any(m['stage'] == 'demand' for _, _, _, m in s):
                r['demand_chats'] += 1
                r['demand_caught'] += int(any(st == 'demand' and p >= 0.5 for _, p, st, _ in s))
            if ch['kind'] == 'slow_burn':
                turn = next(k for k, (_, _, _, m) in enumerate(s) if m['is_scam_msg']); r['slow'] += 1
                r['slow_early'] += int(any(k < turn for k in alerted))
        else:
            r['benign_chats'] += 1; r['benign_alerted'] += int(bool(alerted))
    return dict(scam_chat_recall=round(r['scam_detected'] / max(1, r['scam_chats']), 3),
                demand_stage_caught=round(r['demand_caught'] / max(1, r['demand_chats']), 3),
                benign_chat_false_alarm=round(r['benign_alerted'] / max(1, r['benign_chats']), 3),
                slow_burn_alert_too_early=round(r['slow_early'] / max(1, r['slow']), 3),
                stage_accuracy=round(r['stage_ok'] / max(1, r['stage_n']), 3), n_chats=len(chats))
SUMMARY['M7_test_all'] = conv_metrics(convs['test'])
SUMMARY['M7_test_written_only'] = conv_metrics([c for c in convs['test'] if c.get('source') == 'written'])
print(json.dumps(SUMMARY['M7_test_all'], indent=1)); print(json.dumps(SUMMARY['M7_test_written_only'], indent=1))

In [ ]:
# ---------------- SUMMARY (copy this whole output to Claude) ----------------
json.dump(SUMMARY, open(f'{OUT}/summary_v3.json', 'w'), indent=2)
print(json.dumps(SUMMARY, indent=2))
print('\nFiles to download from', OUT, ':', sorted(os.listdir(OUT)))